In [2]:
#!/usr/bin/env python3

"""
Asymmetric Shapley Values (ASV) & On-Manifold Symmetric Shapley for two datasets
using RandomForestRegressor.

Datasets expected:
  - /mnt/data/cv_risk.csv          (target column: 'cv_risk')
  - /mnt/data/lung_cancer_risk.csv (target column: 'lung_cancer_risk')

What this script does:
- Loads each dataset (drops unnamed/index-like columns automatically)
- Trains a RandomForestRegressor
- Builds an on-manifold value function v_x(S) using a simple conditional KNN imputer:
      v_x(S) = E_{x' | x_S} [ f(x_S ⊕ x'_{~S}) ]
  and baseline v(∅) = E_{x'}[ f(x') ]
- Computes:
  * Symmetric Shapley (uniform permutations) with SAME on-manifold v
  * ASV (distal variant): sample permutations that respect causal edges (ancestors before descendants)
- Aggregates local -> global values across test subset
- Verifies efficiency, saves CSVs & PNG bar charts under ./outputs/

If you know causal relations, populate DATASET_CONSTRAINTS below.
If left empty {}, ASV reduces to symmetric (as expected).
"""

import os
import random
from collections import defaultdict
from typing import Dict, List, Set, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer


RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# Monte Carlo / sampling config
N_PERMUTATIONS_PER_POINT = 64   # how many permutations to sample per point
N_MC_CONDITIONAL = 32           # E[ f | x_S ] samples
N_MC_UNCONDITIONAL = 512        # E[ f ] samples for v(∅)
K_NEIGHBORS = 50                # K for conditional KNN in S-subspace
TEST_SIZE = 0.25

DATASET_CONSTRAINTS: Dict[str, Dict[str, List[str]]] = {
    'cv_risk': {},
    'lung_cancer_risk': {},
}


def drop_unnamed_or_index_cols(df: pd.DataFrame) -> pd.DataFrame:
    drop_cols = []
    for c in df.columns:
        cl = str(c).strip().lower()
        if cl.startswith('unnamed') or cl in {'', 'index', 'id', 'x'}:
            drop_cols.append(c)
    if drop_cols:
        df = df.drop(columns=drop_cols)
    return df

def topological_sort_random(features: List[str],
                            edges: Dict[str, List[str]]) -> List[str]:
    """
    Sample a random topological order consistent with given edges.
    edges: dict ancestor -> list of descendants
    """
    # Build graph
    G = defaultdict(set)
    indeg = defaultdict(int)
    for f in features:
        G[f]  # ensure node exists
        indeg[f]  # ensure indegree exists

    for a, vs in edges.items():
        if a not in G:
            continue
        for v in vs:
            if v in G and v not in G[a]:
                G[a].add(v)
                indeg[v] += 1

    zero = [f for f in features if indeg[f] == 0]
    order = []
    while zero:
        u = random.choice(zero)
        zero.remove(u)
        order.append(u)
        for w in list(G[u]):
            indeg[w] -= 1
            if indeg[w] == 0:
                zero.append(w)
        G[u].clear()

    # If cycle: fall back to random permutation
    if len(order) != len(features):
        order = features[:]
        random.shuffle(order)
    return order

def sample_valid_permutations(features: List[str],
                              edges: Dict[str, List[str]],
                              n: int) -> List[List[str]]:
    return [topological_sort_random(features, edges) for _ in range(n)]

def efficiency_gap_local(phi: np.ndarray, vN: float, v0: float) -> float:
    return abs(phi.sum() - (vN - v0))


class ConditionalKNNImputer:
    """
    Conditional splicing sampler for on-manifold v_x(S).
    Given coalition S (feature indices), we:
      - find K nearest neighbors in the S-subspace (StandardScaler for stability)
      - for each MC sample, pick a neighbor and splice x_S with neighbor's complement
    """
    def __init__(self, X_train: np.ndarray, feature_names: List[str]):
        self.X_train = np.asarray(X_train)
        self.feature_names = feature_names
        self.n, self.d = self.X_train.shape
        self.scaler = StandardScaler().fit(self.X_train)
        self.Xs = self.scaler.transform(self.X_train)

    def sample_spliced(self, x: np.ndarray, S: Set[int], n_mc: int) -> np.ndarray:
        x = np.asarray(x)
        if len(S) == 0:
            # unconditional: draw rows from train (baseline distribution)
            idx = np.random.randint(0, self.n, size=n_mc)
            return self.X_train[idx].copy()

        cols = sorted(list(S))
        Xs_S = self.Xs[:, cols]
        xS = self.scaler.transform(x.reshape(1, -1))[0, cols].reshape(1, -1)

        k = min(K_NEIGHBORS, self.n)
        nbrs = NearestNeighbors(n_neighbors=k, algorithm='auto', metric='euclidean')
        nbrs.fit(Xs_S)
        _, neigh_idx = nbrs.kneighbors(xS, return_distance=True)
        neigh_idx = neigh_idx[0]

        comp = [j for j in range(self.d) if j not in S]
        out = np.tile(x, (n_mc, 1))
        pick = np.random.choice(neigh_idx, size=n_mc, replace=True)
        out[:, comp] = self.X_train[pick][:, comp]
        return out


class OnManifoldValueFunction:
    """
    v_x(S) = E_{x'|x_S}[ f(x_S ⊕ x'_{~S}) ],  v(∅) = E_{x'}[ f(x') ].
    Caches evaluations per (x, S, n_mc) to speed up.
    """
    def __init__(self, model, imputer: ConditionalKNNImputer):
        self.model = model
        self.imputer = imputer
        self.cache = {}

    def _x_key(self, x: np.ndarray) -> bytes:
        # hashable key for caching
        return np.ascontiguousarray(x).tobytes()

    def v_empty(self, n_mc: int = N_MC_UNCONDITIONAL) -> float:
        key = ('__empty__', n_mc)
        if key in self.cache:
            return self.cache[key]
        xs = self.imputer.sample_spliced(x=np.zeros(self.imputer.d), S=set(), n_mc=n_mc)
        pred = float(self.model.predict(xs).mean())
        self.cache[key] = pred
        return pred

    def v(self, x: np.ndarray, S: Set[int], n_mc: int = N_MC_CONDITIONAL) -> float:
        key = (self._x_key(x), tuple(sorted(S)), n_mc)
        if key in self.cache:
            return self.cache[key]
        xs = self.imputer.sample_spliced(x, S, n_mc=n_mc)
        pred = float(self.model.predict(xs).mean())
        self.cache[key] = pred
        return pred


def local_asv_for_point(x: np.ndarray,
                        vfun: OnManifoldValueFunction,
                        feature_names: List[str],
                        constraint_edges: Dict[str, List[str]],
                        n_perms: int = N_PERMUTATIONS_PER_POINT) -> Tuple[np.ndarray, float, float]:
    """
    Local ASV via random valid permutations (ancestors before descendants).
    Returns phi (d,), vN, v0 for efficiency checking.
    """
    d = len(feature_names)
    name_to_idx = {f: i for i, f in enumerate(feature_names)}

    # Sample permutations in name space (respecting edges)
    perms_names = sample_valid_permutations(feature_names, constraint_edges, n_perms)
    perms_idx = [[name_to_idx[nm] for nm in perm] for perm in perms_names]

    v0 = vfun.v_empty()
    vN = vfun.v(x, set(range(d)))
    phi = np.zeros(d, dtype=float)

    for perm in perms_idx:
        S = set()
        prev = v0
        for j in perm:
            curr = vfun.v(x, S | {j})
            phi[j] += (curr - prev)
            prev = curr
            S.add(j)

    phi /= len(perms_idx)
    return phi, vN, v0

def local_symmetric_shap_for_point(x: np.ndarray,
                                   vfun: OnManifoldValueFunction,
                                   d: int,
                                   n_perms: int = N_PERMUTATIONS_PER_POINT) -> Tuple[np.ndarray, float, float]:
    """
    Local symmetric SHAP using uniform random permutations with the SAME on-manifold v.
    """
    v0 = vfun.v_empty()
    vN = vfun.v(x, set(range(d)))
    phi = np.zeros(d, dtype=float)

    for _ in range(n_perms):
        perm = list(range(d))
        random.shuffle(perm)
        S = set()
        prev = v0
        for j in perm:
            curr = vfun.v(x, S | {j})
            phi[j] += (curr - prev)
            prev = curr
            S.add(j)

    phi /= n_perms
    return phi, vN, v0


def train_random_forest_regressor(X_train, y_train) -> RandomForestRegressor:
    rf = RandomForestRegressor(
        n_estimators=400,
        max_features="sqrt",
        random_state=RANDOM_SEED,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)
    return rf

def run_for_dataset(csv_path: str,
                    target_col: str,
                    constraints: Dict[str, List[str]],
                    out_prefix: str):
    print(f"\n=== DATASET: {csv_path} (target={target_col}) ===")
    df = pd.read_csv(csv_path)
    df = drop_unnamed_or_index_cols(df)

    assert target_col in df.columns, f"Target '{target_col}' not found in {csv_path}"
    y = pd.to_numeric(df[target_col], errors='coerce').values.astype(float)
    X_df = df.drop(columns=[target_col]).apply(pd.to_numeric, errors='coerce')
    feature_names = list(X_df.columns)
    X = X_df.values.astype(float)

    # Basic numeric imputation (if any NaNs)
    X = SimpleImputer(strategy='median').fit_transform(X)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED
    )

    model = train_random_forest_regressor(X_train, y_train)
    print(f"Train R^2: {model.score(X_train, y_train):.4f} | Test R^2: {model.score(X_test, y_test):.4f}")

    # On-manifold value function (conditional KNN imputation)
    imputer = ConditionalKNNImputer(X_train, feature_names)
    vfun = OnManifoldValueFunction(model, imputer)
    d = X.shape[1]

    # Evaluate on (subsampled) test points for speed
    MAX_POINTS = 200
    if len(X_test) > MAX_POINTS:
        idx = np.random.choice(len(X_test), size=MAX_POINTS, replace=False)
        X_eval = X_test[idx]
    else:
        X_eval = X_test

    sym_vals = []
    asv_vals = []
    eff_gaps_sym = []
    eff_gaps_asv = []

    for i, x in enumerate(X_eval):
        # symmetric (on-manifold v)
        phi_sym, vN, v0 = local_symmetric_shap_for_point(x, vfun, d)
        sym_vals.append(phi_sym)
        eff_gaps_sym.append(efficiency_gap_local(phi_sym, vN, v0))

        # ASV (distal) with constraints
        phi_asv, vN2, v0_2 = local_asv_for_point(x, vfun, feature_names, constraints)
        asv_vals.append(phi_asv)
        eff_gaps_asv.append(efficiency_gap_local(phi_asv, vN2, v0_2))

    sym_vals = np.vstack(sym_vals)
    asv_vals = np.vstack(asv_vals)
    global_sym = sym_vals.mean(axis=0)
    global_asv = asv_vals.mean(axis=0)

    print(f"Mean efficiency gap — Symmetric: {np.mean(eff_gaps_sym):.6f} | ASV: {np.mean(eff_gaps_asv):.6f}")

    # Save outputs
    os.makedirs("outputs", exist_ok=True)

    df_sym = pd.DataFrame({'feature': feature_names, 'global_value': global_sym})
    df_asv = pd.DataFrame({'feature': feature_names, 'global_value': global_asv})

    sym_csv = f"outputs/{out_prefix}_global_symmetric_onmanifold.csv"
    asv_csv = f"outputs/{out_prefix}_global_asv_onmanifold.csv"
    df_sym.to_csv(sym_csv, index=False)
    df_asv.to_csv(asv_csv, index=False)
    print(f"Saved: {sym_csv}")
    print(f"Saved: {asv_csv}")

    # Bar plots
    def plot_bar(values, fname, title):
        order = np.argsort(values)[::-1]
        plt.figure(figsize=(9, 5))
        plt.bar(np.array(feature_names)[order], values[order])
        plt.xticks(rotation=60, ha='right')
        plt.ylabel("Global value")
        plt.title(title)
        plt.tight_layout()
        plt.savefig(fname, dpi=150)
        plt.close()

    sym_png = f"outputs/{out_prefix}_global_symmetric_onmanifold.png"
    asv_png = f"outputs/{out_prefix}_global_asv_onmanifold.png"
    plot_bar(global_sym, sym_png, f"{out_prefix}: Global Symmetric (on-manifold)")
    plot_bar(global_asv, asv_png, f"{out_prefix}: Global ASV (on-manifold)")
    print(f"Saved: {sym_png}")
    print(f"Saved: {asv_png}")

    # Console summary
    print(f"\nTop features (Symmetric on-manifold):")
    for i in np.argsort(global_sym)[::-1]:
        print(f"  {feature_names[i]:25s} {global_sym[i]: .6f}")
    print(f"\nTop features (ASV on-manifold):")
    for i in np.argsort(global_asv)[::-1]:
        print(f"  {feature_names[i]:25s} {global_asv[i]: .6f}")


def main():
    tasks = [
        ("../../datasets/cv_risk.csv", "cv_risk",
         DATASET_CONSTRAINTS.get('cv_risk', {}), "cv_risk_rf"),
        ("../../datasets/lung_cancer_risk.csv", "lung_cancer_risk",
         DATASET_CONSTRAINTS.get('lung_cancer_risk', {}), "lung_cancer_rf"),
    ]
    for csv_path, target, constraints, prefix in tasks:
        run_for_dataset(csv_path, target, constraints, prefix)

if __name__ == "__main__":
    main()


=== DATASET: ../datasets/cv_risk.csv (target=cv_risk) ===
Train R^2: 0.9542 | Test R^2: 0.6508
Mean efficiency gap — Symmetric: 0.000000 | ASV: 0.000000
Saved: outputs/cv_risk_rf_global_symmetric_onmanifold.csv
Saved: outputs/cv_risk_rf_global_asv_onmanifold.csv
Saved: outputs/cv_risk_rf_global_symmetric_onmanifold.png
Saved: outputs/cv_risk_rf_global_asv_onmanifold.png

Top features (Symmetric on-manifold):
  family_history             0.022977
  diet_score                -0.017713
  sleep_duration            -0.045485
  bmi                       -0.090604

Top features (ASV on-manifold):
  family_history             0.020955
  diet_score                -0.007345
  sleep_duration            -0.050722
  bmi                       -0.093715

=== DATASET: ../datasets/lung_cancer_risk.csv (target=lung_cancer_risk) ===
Train R^2: 0.9648 | Test R^2: 0.7304
Mean efficiency gap — Symmetric: 0.000000 | ASV: 0.000000
Saved: outputs/lung_cancer_rf_global_symmetric_onmanifold.csv
Saved: outputs/l

In [4]:
#!/usr/bin/env python3

"""
Perfect Paper-Faithful Asymmetric Shapley Values (ASV) Implementation
Based on arXiv:1910.06358v3 - Frye, Feige, & Rowat (2020)

This implementation fixes all the issues in the original code to be truly paper-faithful:
1. Proper causal order filtering (distal constraint)
2. Correct conditional expectations (on-manifold)
3. Fair comparison (same imputer for both methods)
4. Global aggregation with efficiency checks
5. Proper topological sampling (not factorial enumeration)
6. Support for both regression and classification value functions
"""

import os
import random
from collections import defaultdict
from typing import Dict, List, Set, Tuple, Optional
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score


RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# Monte Carlo / sampling config
N_PERMUTATIONS_PER_POINT = 100   # how many permutations to sample per point
N_MC_CONDITIONAL = 50           # E[ f | x_S ] samples
N_MC_UNCONDITIONAL = 200        # E[ f ] samples for v(∅)
K_NEIGHBORS = 20                # K for conditional KNN in S-subspace
TEST_SIZE = 0.25
MAX_TEST_POINTS = 50            # Limit for computational efficiency

# Causal constraints (distal ASV: ancestors must precede descendants)
DATASET_CONSTRAINTS: Dict[str, Dict[str, List[str]]] = {
    'cv_risk': {
        'family_history': ['bmi', 'diet_score'],
        'bmi': ['sleep_duration']
    },
    'lung_cancer_risk': {
        'smoking': ['stress', 'drink_coffee']
    },
}


def drop_unnamed_or_index_cols(df: pd.DataFrame) -> pd.DataFrame:
    """Remove unnamed or index-like columns"""
    drop_cols = []
    for c in df.columns:
        cl = str(c).strip().lower()
        if cl.startswith('unnamed') or cl in {'', 'index', 'id', 'x'}:
            drop_cols.append(c)
    if drop_cols:
        df = df.drop(columns=drop_cols)
    return df

def create_causal_dag(feature_names: List[str], edges: Dict[str, List[str]]) -> nx.DiGraph:
    """Create causal DAG from feature names and edges"""
    G = nx.DiGraph()
    G.add_nodes_from(feature_names)

    for cause, effects in edges.items():
        if cause in feature_names:
            for effect in effects:
                if effect in feature_names:
                    G.add_edge(cause, effect)

    if not nx.is_directed_acyclic_graph(G):
        raise ValueError("Causal relationships must form a DAG (no cycles)")

    return G

def sample_topological_orders(G: nx.DiGraph, n_samples: int) -> List[List[str]]:
    """
    Sample n_samples random topological orders using Kahn's algorithm with random tie-breaking
    This is more efficient than enumerating all permutations
    """
    orders = []

    for _ in range(n_samples):
        # Kahn's algorithm with random tie-breaking
        in_degree = {node: G.in_degree(node) for node in G.nodes()}
        queue = [node for node in G.nodes() if in_degree[node] == 0]
        order = []

        while queue:
            # Random tie-breaking
            random.shuffle(queue)
            node = queue.pop(0)
            order.append(node)

            # Update in-degrees of neighbors
            for neighbor in G.neighbors(node):
                in_degree[neighbor] -= 1
                if in_degree[neighbor] == 0:
                    queue.append(neighbor)

        orders.append(order)

    return orders

def efficiency_gap_local(phi: np.ndarray, vN: float, v0: float) -> float:
    """Check efficiency property: sum(phi) should equal v(N) - v(∅)"""
    return abs(phi.sum() - (vN - v0))


class ConditionalKNNImputer:
    """
    Conditional splicing sampler for on-manifold v_x(S).
    This implements the paper's on-manifold value function using KNN-conditional sampling.
    """
    def __init__(self, X_train: np.ndarray, feature_names: List[str]):
        self.X_train = np.asarray(X_train)
        self.feature_names = feature_names
        self.n, self.d = self.X_train.shape
        self.scaler = StandardScaler().fit(self.X_train)
        self.Xs = self.scaler.transform(self.X_train)

    def sample_conditional(self, x: np.ndarray, S: Set[int], n_mc: int) -> np.ndarray:
        """
        Sample from p(x'_{~S} | x_S) using KNN-conditional sampling
        This implements the paper's on-manifold conditional expectation
        """
        x = np.asarray(x)
        if len(S) == 0:
            # Unconditional: sample from training data
            idx = np.random.randint(0, self.n, size=n_mc)
            return self.X_train[idx].copy()

        # Find k-nearest neighbors in the S-subspace
        cols = sorted(list(S))
        Xs_S = self.Xs[:, cols]
        xS = self.scaler.transform(x.reshape(1, -1))[0, cols].reshape(1, -1)

        k = min(K_NEIGHBORS, self.n)
        nbrs = NearestNeighbors(n_neighbors=k, algorithm='auto', metric='euclidean')
        nbrs.fit(Xs_S)
        _, neigh_idx = nbrs.kneighbors(xS, return_distance=True)
        neigh_idx = neigh_idx[0]

        # Sample from neighbors and condition on S
        comp = [j for j in range(self.d) if j not in S]
        out = np.tile(x, (n_mc, 1))
        pick = np.random.choice(neigh_idx, size=n_mc, replace=True)
        out[:, comp] = self.X_train[pick][:, comp]
        return out


class OnManifoldValueFunction:
    """
    On-manifold value function v_x(S) = E_{x'|x_S}[f(x_S ⊕ x'_{~S})]
    This implements Equation (4) from the paper exactly
    """
    def __init__(self, model, imputer: ConditionalKNNImputer, task_type: str = 'regression'):
        self.model = model
        self.imputer = imputer
        self.task_type = task_type
        self.cache = {}

    def _cache_key(self, x: np.ndarray, S: Set[int], n_mc: int) -> str:
        """Create cache key for memoization"""
        return f"{hash(x.tobytes())}_{tuple(sorted(S))}_{n_mc}"

    def v_empty(self, n_mc: int = N_MC_UNCONDITIONAL) -> float:
        """v(∅) = E_{x'}[f(x')] - baseline expectation"""
        key = self._cache_key(np.zeros(self.imputer.d), set(), n_mc)
        if key in self.cache:
            return self.cache[key]

        xs = self.imputer.sample_conditional(x=np.zeros(self.imputer.d), S=set(), n_mc=n_mc)

        if self.task_type == 'regression':
            pred = float(self.model.predict(xs).mean())
        else:  # classification
            pred = float(self.model.predict_proba(xs).mean())

        self.cache[key] = pred
        return pred

    def v(self, x: np.ndarray, S: Set[int], n_mc: int = N_MC_CONDITIONAL) -> float:
        """v(S) = E_{x'|x_S}[f(x_S ⊕ x'_{~S})] - conditional expectation"""
        key = self._cache_key(x, S, n_mc)
        if key in self.cache:
            return self.cache[key]

        xs = self.imputer.sample_conditional(x, S, n_mc=n_mc)

        if self.task_type == 'regression':
            pred = float(self.model.predict(xs).mean())
        else:  # classification
            pred = float(self.model.predict_proba(xs).mean())

        self.cache[key] = pred
        return pred


def local_asv_for_point(x: np.ndarray,
                        vfun: OnManifoldValueFunction,
                        feature_names: List[str],
                        G: nx.DiGraph,
                        n_perms: int = N_PERMUTATIONS_PER_POINT) -> Tuple[np.ndarray, float, float]:
    """
    Local ASV using order-restricted permutations (distal constraint)
    This implements Equation (10) from the paper exactly
    """
    d = len(feature_names)
    name_to_idx = {f: i for i, f in enumerate(feature_names)}

    # Sample topological orders consistent with causal DAG
    perms_names = sample_topological_orders(G, n_perms)
    perms_idx = [[name_to_idx[nm] for nm in perm] for perm in perms_names]

    v0 = vfun.v_empty()
    vN = vfun.v(x, set(range(d)))
    phi = np.zeros(d, dtype=float)

    # For each valid permutation π
    for perm in perms_idx:
        S = set()
        prev = v0
        for j in perm:
            curr = vfun.v(x, S | {j})
            phi[j] += (curr - prev)  # Marginal contribution
            prev = curr
            S.add(j)

    phi /= len(perms_idx)  # Average over all valid permutations
    return phi, vN, v0

def local_symmetric_shap_for_point(x: np.ndarray,
                                   vfun: OnManifoldValueFunction,
                                   d: int,
                                   n_perms: int = N_PERMUTATIONS_PER_POINT) -> Tuple[np.ndarray, float, float]:
    """
    Local symmetric SHAP using uniform random permutations with the SAME on-manifold v
    This ensures fair comparison with ASV
    """
    v0 = vfun.v_empty()
    vN = vfun.v(x, set(range(d)))
    phi = np.zeros(d, dtype=float)

    for _ in range(n_perms):
        perm = list(range(d))
        random.shuffle(perm)  # Uniform random permutation
        S = set()
        prev = v0
        for j in perm:
            curr = vfun.v(x, S | {j})
            phi[j] += (curr - prev)  # Marginal contribution
            prev = curr
            S.add(j)

    phi /= n_perms  # Average over all permutations
    return phi, vN, v0


def train_model(X_train, y_train, task_type: str = 'regression'):
    """Train Random Forest model based on task type"""
    if task_type == 'regression':
        model = RandomForestRegressor(
            n_estimators=200,
            max_features="sqrt",
            random_state=RANDOM_SEED,
            n_jobs=-1
        )
    else:  # classification
        model = RandomForestClassifier(
            n_estimators=200,
            max_features="sqrt",
            random_state=RANDOM_SEED,
            n_jobs=-1
        )

    model.fit(X_train, y_train)
    return model

def run_for_dataset(csv_path: str,
                    target_col: str,
                    constraints: Dict[str, List[str]],
                    out_prefix: str,
                    task_type: str = 'regression'):
    """Run analysis for a single dataset"""
    print(f"\n=== DATASET: {csv_path} (target={target_col}, task={task_type}) ===")

    # Load and prepare data
    df = pd.read_csv(csv_path)
    df = drop_unnamed_or_index_cols(df)

    assert target_col in df.columns, f"Target '{target_col}' not found in {csv_path}"
    y = pd.to_numeric(df[target_col], errors='coerce').values.astype(float)
    X_df = df.drop(columns=[target_col]).apply(pd.to_numeric, errors='coerce')
    feature_names = list(X_df.columns)
    X = X_df.values.astype(float)

    # Basic numeric imputation
    X = SimpleImputer(strategy='median').fit_transform(X)

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED
    )

    # Train model
    model = train_model(X_train, y_train, task_type)

    if task_type == 'regression':
        train_score = model.score(X_train, y_train)
        test_score = model.score(X_test, y_test)
        print(f"Train R^2: {train_score:.4f} | Test R^2: {test_score:.4f}")
    else:
        train_score = model.score(X_train, y_train)
        test_score = model.score(X_test, y_test)
        print(f"Train Accuracy: {train_score:.4f} | Test Accuracy: {test_score:.4f}")

    # Create causal DAG
    G = create_causal_dag(feature_names, constraints)
    print(f"Causal DAG edges: {list(G.edges())}")

    # On-manifold value function (same imputer for both methods)
    imputer = ConditionalKNNImputer(X_train, feature_names)
    vfun = OnManifoldValueFunction(model, imputer, task_type)
    d = X.shape[1]

    # Evaluate on test points
    n_eval = min(MAX_TEST_POINTS, len(X_test))
    if len(X_test) > n_eval:
        idx = np.random.choice(len(X_test), size=n_eval, replace=False)
        X_eval = X_test[idx]
    else:
        X_eval = X_test

    print(f"Evaluating on {len(X_eval)} test points...")

    # Compute local explanations
    sym_vals = []
    asv_vals = []
    eff_gaps_sym = []
    eff_gaps_asv = []

    for i, x in enumerate(X_eval):
        if i % 10 == 0:
            print(f"  Processing point {i}/{len(X_eval)}")

        # Symmetric SHAP (on-manifold)
        phi_sym, vN, v0 = local_symmetric_shap_for_point(x, vfun, d)
        sym_vals.append(phi_sym)
        eff_gaps_sym.append(efficiency_gap_local(phi_sym, vN, v0))

        # ASV (distal) with constraints
        phi_asv, vN2, v0_2 = local_asv_for_point(x, vfun, feature_names, G)
        asv_vals.append(phi_asv)
        eff_gaps_asv.append(efficiency_gap_local(phi_asv, vN2, v0_2))

    # Aggregate to global values
    sym_vals = np.vstack(sym_vals)
    asv_vals = np.vstack(asv_vals)
    global_sym = sym_vals.mean(axis=0)
    global_asv = asv_vals.mean(axis=0)
    global_sym_std = sym_vals.std(axis=0)
    global_asv_std = asv_vals.std(axis=0)

    print(f"Mean efficiency gap — Symmetric: {np.mean(eff_gaps_sym):.6f} | ASV: {np.mean(eff_gaps_asv):.6f}")

    # Save outputs
    os.makedirs("outputs", exist_ok=True)

    # Save results
    df_sym = pd.DataFrame({
        'feature': feature_names,
        'global_value': global_sym,
        'std': global_sym_std
    })
    df_asv = pd.DataFrame({
        'feature': feature_names,
        'global_value': global_asv,
        'std': global_asv_std
    })

    sym_csv = f"outputs/{out_prefix}_global_symmetric_onmanifold.csv"
    asv_csv = f"outputs/{out_prefix}_global_asv_onmanifold.csv"
    df_sym.to_csv(sym_csv, index=False)
    df_asv.to_csv(asv_csv, index=False)
    print(f"Saved: {sym_csv}")
    print(f"Saved: {asv_csv}")

    # Create comparison plots
    def plot_comparison(values_sym, values_asv, std_sym, std_asv, fname, title):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

        # Symmetric plot
        order = np.argsort(values_sym)[::-1]
        ax1.bar(np.array(feature_names)[order], values_sym[order],
                yerr=std_sym[order], capsize=5, alpha=0.7)
        ax1.set_xticklabels(np.array(feature_names)[order], rotation=45, ha='right')
        ax1.set_ylabel("Global Value")
        ax1.set_title(f"Symmetric (on-manifold)")
        ax1.grid(True, alpha=0.3)

        # ASV plot
        order = np.argsort(values_asv)[::-1]
        ax2.bar(np.array(feature_names)[order], values_asv[order],
                yerr=std_asv[order], capsize=5, alpha=0.7)
        ax2.set_xticklabels(np.array(feature_names)[order], rotation=45, ha='right')
        ax2.set_ylabel("Global Value")
        ax2.set_title(f"ASV (on-manifold)")
        ax2.grid(True, alpha=0.3)

        plt.suptitle(title)
        plt.tight_layout()
        plt.savefig(fname, dpi=150, bbox_inches='tight')
        plt.close()

    comparison_png = f"outputs/{out_prefix}_comparison_onmanifold.png"
    plot_comparison(global_sym, global_asv, global_sym_std, global_asv_std,
                   comparison_png, f"{out_prefix}: On-Manifold Comparison")
    print(f"Saved: {comparison_png}")

    # Console summary
    print(f"\nTop features (Symmetric on-manifold):")
    for i in np.argsort(global_sym)[::-1]:
        print(f"  {feature_names[i]:25s} {global_sym[i]: .6f} ± {global_sym_std[i]:.6f}")

    print(f"\nTop features (ASV on-manifold):")
    for i in np.argsort(global_asv)[::-1]:
        print(f"  {feature_names[i]:25s} {global_asv[i]: .6f} ± {global_asv_std[i]:.6f}")


def main():
    """Main execution function"""
    print("Perfect Paper-Faithful Asymmetric Shapley Values (ASV) Implementation")
    print("Based on arXiv:1910.06358v3 - Frye, Feige, & Rowat (2020)")
    print("=" * 80)

    # Define datasets to analyze
    tasks = [
        ("../../datasets/lung_cancer_risk.csv", "lung_cancer_risk",
         DATASET_CONSTRAINTS.get('lung_cancer_risk', {}), "lung_cancer_rf", "regression"),
        ("../../datasets/cv_risk.csv", "cv_risk",
         DATASET_CONSTRAINTS.get('cv_risk', {}), "cv_risk_rf", "regression"),
    ]

    for csv_path, target, constraints, prefix, task_type in tasks:
        if os.path.exists(csv_path):
            run_for_dataset(csv_path, target, constraints, prefix, task_type)
        else:
            print(f"Warning: {csv_path} not found, skipping...")

if __name__ == "__main__":
    main()

Perfect Paper-Faithful Asymmetric Shapley Values (ASV) Implementation
Based on arXiv:1910.06358v3 - Frye, Feige, & Rowat (2020)

=== DATASET: ../datasets/lung_cancer_risk.csv (target=lung_cancer_risk, task=regression) ===
Train R^2: 0.9643 | Test R^2: 0.7291
Causal DAG edges: [('smoking', 'stress'), ('smoking', 'drink_coffee')]
Evaluating on 50 test points...
  Processing point 0/50
  Processing point 10/50
  Processing point 20/50
  Processing point 30/50
  Processing point 40/50
Mean efficiency gap — Symmetric: 0.000000 | ASV: 0.000000
Saved: outputs/lung_cancer_rf_global_symmetric_onmanifold.csv
Saved: outputs/lung_cancer_rf_global_asv_onmanifold.csv


Saved: outputs/lung_cancer_rf_comparison_onmanifold.png

Top features (Symmetric on-manifold):
  drink_coffee              -0.178566 ± 1.775114
  stress                    -0.209363 ± 0.874311
  smoking                   -0.385857 ± 1.478393

Top features (ASV on-manifold):
  drink_coffee               0.055799 ± 0.964613
  stress                    -0.229514 ± 1.006356
  smoking                   -0.600071 ± 2.634941

=== DATASET: ../datasets/cv_risk.csv (target=cv_risk, task=regression) ===
Train R^2: 0.9540 | Test R^2: 0.6498
Causal DAG edges: [('family_history', 'bmi'), ('family_history', 'diet_score'), ('bmi', 'sleep_duration')]
Evaluating on 50 test points...
  Processing point 0/50
  Processing point 10/50
  Processing point 20/50
  Processing point 30/50
  Processing point 40/50
Mean efficiency gap — Symmetric: 0.000000 | ASV: 0.000000
Saved: outputs/cv_risk_rf_global_symmetric_onmanifold.csv
Saved: outputs/cv_risk_rf_global_asv_onmanifold.csv


Saved: outputs/cv_risk_rf_comparison_onmanifold.png

Top features (Symmetric on-manifold):
  sleep_duration             0.188351 ± 0.677624
  family_history             0.032575 ± 0.266325
  diet_score                -0.092449 ± 0.709293
  bmi                       -0.162301 ± 1.794317

Top features (ASV on-manifold):
  family_history             0.121539 ± 0.598569
  sleep_duration             0.036871 ± 0.321621
  bmi                       -0.086897 ± 2.135618
  diet_score                -0.105337 ± 0.790482


In [5]:
#!/usr/bin/env python3

"""
Perfect Paper-Faithful Asymmetric Shapley Values (ASV) Implementation
Based on arXiv:1910.06358v3 - Frye, Feige, & Rowat (2020)

This implementation fixes all the issues in the original code to be truly paper-faithful:
1. Proper causal order filtering (distal constraint)
2. Correct conditional expectations (on-manifold)
3. Fair comparison (same imputer for both methods)
4. Global aggregation with efficiency checks
5. Proper topological sampling (not factorial enumeration)
6. Support for both regression and classification value functions
"""

import os
import random
from collections import defaultdict
from typing import Dict, List, Set, Tuple, Optional
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score


RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# Monte Carlo / sampling config
N_PERMUTATIONS_PER_POINT = 100   # how many permutations to sample per point
N_MC_CONDITIONAL = 50           # E[ f | x_S ] samples
N_MC_UNCONDITIONAL = 200        # E[ f ] samples for v(∅)
K_NEIGHBORS = 20                # K for conditional KNN in S-subspace
TEST_SIZE = 0.25
MAX_TEST_POINTS = 50            # Limit for computational efficiency

# Causal constraints (distal ASV: ancestors must precede descendants)
DATASET_CONSTRAINTS: Dict[str, Dict[str, List[str]]] = {
    'cv_risk': {
        'family_history': ['bmi', 'diet_score'],
        'bmi': ['sleep_duration']
    },
    'lung_cancer_risk': {
        'smoking': ['stress', 'drink_coffee']
    },
}


def drop_unnamed_or_index_cols(df: pd.DataFrame) -> pd.DataFrame:
    """Remove unnamed or index-like columns"""
    drop_cols = []
    for c in df.columns:
        cl = str(c).strip().lower()
        if cl.startswith('unnamed') or cl in {'', 'index', 'id', 'x'}:
            drop_cols.append(c)
    if drop_cols:
        df = df.drop(columns=drop_cols)
    return df

def create_causal_dag(feature_names: List[str], edges: Dict[str, List[str]]) -> nx.DiGraph:
    """Create causal DAG from feature names and edges"""
    G = nx.DiGraph()
    G.add_nodes_from(feature_names)

    for cause, effects in edges.items():
        if cause in feature_names:
            for effect in effects:
                if effect in feature_names:
                    G.add_edge(cause, effect)

    if not nx.is_directed_acyclic_graph(G):
        raise ValueError("Causal relationships must form a DAG (no cycles)")

    return G

def sample_topological_orders(G: nx.DiGraph, n_samples: int) -> List[List[str]]:
    """
    Sample n_samples random topological orders using Kahn's algorithm with random tie-breaking
    This is more efficient than enumerating all permutations
    """
    orders = []

    for _ in range(n_samples):
        # Kahn's algorithm with random tie-breaking
        in_degree = {node: G.in_degree(node) for node in G.nodes()}
        queue = [node for node in G.nodes() if in_degree[node] == 0]
        order = []

        while queue:
            # Random tie-breaking
            random.shuffle(queue)
            node = queue.pop(0)
            order.append(node)

            # Update in-degrees of neighbors
            for neighbor in G.neighbors(node):
                in_degree[neighbor] -= 1
                if in_degree[neighbor] == 0:
                    queue.append(neighbor)

        orders.append(order)

    return orders

def efficiency_gap_local(phi: np.ndarray, vN: float, v0: float) -> float:
    """Check efficiency property: sum(phi) should equal v(N) - v(∅)"""
    return abs(phi.sum() - (vN - v0))


class ConditionalKNNImputer:
    """
    Conditional splicing sampler for on-manifold v_x(S).
    This implements the paper's on-manifold value function using KNN-conditional sampling.
    """
    def __init__(self, X_train: np.ndarray, feature_names: List[str]):
        self.X_train = np.asarray(X_train)
        self.feature_names = feature_names
        self.n, self.d = self.X_train.shape
        self.scaler = StandardScaler().fit(self.X_train)
        self.Xs = self.scaler.transform(self.X_train)

    def sample_conditional(self, x: np.ndarray, S: Set[int], n_mc: int) -> np.ndarray:
        """
        Sample from p(x'_{~S} | x_S) using KNN-conditional sampling
        This implements the paper's on-manifold conditional expectation
        """
        x = np.asarray(x)
        if len(S) == 0:
            # Unconditional: sample from training data
            idx = np.random.randint(0, self.n, size=n_mc)
            return self.X_train[idx].copy()

        # Find k-nearest neighbors in the S-subspace
        cols = sorted(list(S))
        Xs_S = self.Xs[:, cols]
        xS = self.scaler.transform(x.reshape(1, -1))[0, cols].reshape(1, -1)

        k = min(K_NEIGHBORS, self.n)
        nbrs = NearestNeighbors(n_neighbors=k, algorithm='auto', metric='euclidean')
        nbrs.fit(Xs_S)
        _, neigh_idx = nbrs.kneighbors(xS, return_distance=True)
        neigh_idx = neigh_idx[0]

        # Sample from neighbors and condition on S
        comp = [j for j in range(self.d) if j not in S]
        out = np.tile(x, (n_mc, 1))
        pick = np.random.choice(neigh_idx, size=n_mc, replace=True)
        out[:, comp] = self.X_train[pick][:, comp]
        return out


class OnManifoldValueFunction:
    """
    On-manifold value function v_x(S) = E_{x'|x_S}[f(x_S ⊕ x'_{~S})]
    This implements Equation (4) from the paper exactly
    """
    def __init__(self, model, imputer: ConditionalKNNImputer, task_type: str = 'regression'):
        self.model = model
        self.imputer = imputer
        self.task_type = task_type
        self.cache = {}

    def _cache_key(self, x: np.ndarray, S: Set[int], n_mc: int) -> str:
        """Create cache key for memoization"""
        return f"{hash(x.tobytes())}_{tuple(sorted(S))}_{n_mc}"

    def v_empty(self, n_mc: int = N_MC_UNCONDITIONAL) -> float:
        """v(∅) = E_{x'}[f(x')] - baseline expectation"""
        key = self._cache_key(np.zeros(self.imputer.d), set(), n_mc)
        if key in self.cache:
            return self.cache[key]

        xs = self.imputer.sample_conditional(x=np.zeros(self.imputer.d), S=set(), n_mc=n_mc)

        if self.task_type == 'regression':
            pred = float(self.model.predict(xs).mean())
        else:  # classification
            pred = float(self.model.predict_proba(xs).mean())

        self.cache[key] = pred
        return pred

    def v(self, x: np.ndarray, S: Set[int], n_mc: int = N_MC_CONDITIONAL) -> float:
        """v(S) = E_{x'|x_S}[f(x_S ⊕ x'_{~S})] - conditional expectation"""
        key = self._cache_key(x, S, n_mc)
        if key in self.cache:
            return self.cache[key]

        xs = self.imputer.sample_conditional(x, S, n_mc=n_mc)

        if self.task_type == 'regression':
            pred = float(self.model.predict(xs).mean())
        else:  # classification
            pred = float(self.model.predict_proba(xs).mean())

        self.cache[key] = pred
        return pred


def local_asv_for_point(x: np.ndarray,
                        vfun: OnManifoldValueFunction,
                        feature_names: List[str],
                        G: nx.DiGraph,
                        n_perms: int = N_PERMUTATIONS_PER_POINT) -> Tuple[np.ndarray, float, float]:
    """
    Local ASV using order-restricted permutations (distal constraint)
    This implements Equation (10) from the paper exactly
    """
    d = len(feature_names)
    name_to_idx = {f: i for i, f in enumerate(feature_names)}

    # Sample topological orders consistent with causal DAG
    perms_names = sample_topological_orders(G, n_perms)
    perms_idx = [[name_to_idx[nm] for nm in perm] for perm in perms_names]

    v0 = vfun.v_empty()
    vN = vfun.v(x, set(range(d)))
    phi = np.zeros(d, dtype=float)

    # For each valid permutation π
    for perm in perms_idx:
        S = set()
        prev = v0
        for j in perm:
            curr = vfun.v(x, S | {j})
            phi[j] += (curr - prev)  # Marginal contribution
            prev = curr
            S.add(j)

    phi /= len(perms_idx)  # Average over all valid permutations
    return phi, vN, v0

def local_symmetric_shap_for_point(x: np.ndarray,
                                   vfun: OnManifoldValueFunction,
                                   d: int,
                                   n_perms: int = N_PERMUTATIONS_PER_POINT) -> Tuple[np.ndarray, float, float]:
    """
    Local symmetric SHAP using uniform random permutations with the SAME on-manifold v
    This ensures fair comparison with ASV
    """
    v0 = vfun.v_empty()
    vN = vfun.v(x, set(range(d)))
    phi = np.zeros(d, dtype=float)

    for _ in range(n_perms):
        perm = list(range(d))
        random.shuffle(perm)  # Uniform random permutation
        S = set()
        prev = v0
        for j in perm:
            curr = vfun.v(x, S | {j})
            phi[j] += (curr - prev)  # Marginal contribution
            prev = curr
            S.add(j)

    phi /= n_perms  # Average over all permutations
    return phi, vN, v0


def train_model(X_train, y_train, task_type: str = 'regression'):
    """Train Random Forest model based on task type"""
    if task_type == 'regression':
        model = RandomForestRegressor(
            n_estimators=200,
            max_features="sqrt",
            random_state=RANDOM_SEED,
            n_jobs=-1
        )
    else:  # classification
        model = RandomForestClassifier(
            n_estimators=200,
            max_features="sqrt",
            random_state=RANDOM_SEED,
            n_jobs=-1
        )

    model.fit(X_train, y_train)
    return model

def run_for_dataset(csv_path: str,
                    target_col: str,
                    constraints: Dict[str, List[str]],
                    out_prefix: str,
                    task_type: str = 'regression'):
    """Run analysis for a single dataset"""
    print(f"\n=== DATASET: {csv_path} (target={target_col}, task={task_type}) ===")

    # Load and prepare data
    df = pd.read_csv(csv_path)
    df = drop_unnamed_or_index_cols(df)

    assert target_col in df.columns, f"Target '{target_col}' not found in {csv_path}"
    y = pd.to_numeric(df[target_col], errors='coerce').values.astype(float)
    X_df = df.drop(columns=[target_col]).apply(pd.to_numeric, errors='coerce')
    feature_names = list(X_df.columns)
    X = X_df.values.astype(float)

    # Basic numeric imputation
    X = SimpleImputer(strategy='median').fit_transform(X)

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED
    )

    # Train model
    model = train_model(X_train, y_train, task_type)

    if task_type == 'regression':
        train_score = model.score(X_train, y_train)
        test_score = model.score(X_test, y_test)
        print(f"Train R^2: {train_score:.4f} | Test R^2: {test_score:.4f}")
    else:
        train_score = model.score(X_train, y_train)
        test_score = model.score(X_test, y_test)
        print(f"Train Accuracy: {train_score:.4f} | Test Accuracy: {test_score:.4f}")

    # Create causal DAG
    G = create_causal_dag(feature_names, constraints)
    print(f"Causal DAG edges: {list(G.edges())}")

    # On-manifold value function (same imputer for both methods)
    imputer = ConditionalKNNImputer(X_train, feature_names)
    vfun = OnManifoldValueFunction(model, imputer, task_type)
    d = X.shape[1]

    # Evaluate on test points
    n_eval = min(MAX_TEST_POINTS, len(X_test))
    if len(X_test) > n_eval:
        idx = np.random.choice(len(X_test), size=n_eval, replace=False)
        X_eval = X_test[idx]
    else:
        X_eval = X_test

    print(f"Evaluating on {len(X_eval)} test points...")

    # Compute local explanations
    sym_vals = []
    asv_vals = []
    eff_gaps_sym = []
    eff_gaps_asv = []

    for i, x in enumerate(X_eval):
        if i % 10 == 0:
            print(f"  Processing point {i}/{len(X_eval)}")

        # Symmetric SHAP (on-manifold)
        phi_sym, vN, v0 = local_symmetric_shap_for_point(x, vfun, d)
        sym_vals.append(phi_sym)
        eff_gaps_sym.append(efficiency_gap_local(phi_sym, vN, v0))

        # ASV (distal) with constraints
        phi_asv, vN2, v0_2 = local_asv_for_point(x, vfun, feature_names, G)
        asv_vals.append(phi_asv)
        eff_gaps_asv.append(efficiency_gap_local(phi_asv, vN2, v0_2))

    # Aggregate to global values
    sym_vals = np.vstack(sym_vals)
    asv_vals = np.vstack(asv_vals)
    global_sym = sym_vals.mean(axis=0)
    global_asv = asv_vals.mean(axis=0)
    global_sym_std = sym_vals.std(axis=0)
    global_asv_std = asv_vals.std(axis=0)

    print(f"Mean efficiency gap — Symmetric: {np.mean(eff_gaps_sym):.6f} | ASV: {np.mean(eff_gaps_asv):.6f}")

    # Save outputs
    os.makedirs("outputs", exist_ok=True)

    # Save results
    df_sym = pd.DataFrame({
        'feature': feature_names,
        'global_value': global_sym,
        'std': global_sym_std
    })
    df_asv = pd.DataFrame({
        'feature': feature_names,
        'global_value': global_asv,
        'std': global_asv_std
    })

    sym_csv = f"outputs/{out_prefix}_global_symmetric_onmanifold.csv"
    asv_csv = f"outputs/{out_prefix}_global_asv_onmanifold.csv"
    df_sym.to_csv(sym_csv, index=False)
    df_asv.to_csv(asv_csv, index=False)
    print(f"Saved: {sym_csv}")
    print(f"Saved: {asv_csv}")

    def plot_single_bar(values, std, fname, title):
        order = np.argsort(values)[::-1]
        plt.figure(figsize=(9, 5))
        plt.bar(np.array(feature_names)[order], values[order], yerr=std[order], capsize=5, alpha=0.8)
        plt.xticks(rotation=60, ha='right')
        plt.ylabel("Global Value")
        plt.title(title)
        plt.tight_layout()
        plt.savefig(fname, dpi=150)
        plt.close()

    sym_png = f"outputs/{out_prefix}_global_symmetric_onmanifold.png"
    asv_png = f"outputs/{out_prefix}_global_asv_onmanifold.png"
    plot_single_bar(global_sym, global_sym_std, sym_png, f"{out_prefix}: Global Symmetric (on-manifold)")
    plot_single_bar(global_asv, global_asv_std, asv_png, f"{out_prefix}: Global ASV (on-manifold)")
    print(f"Saved: {sym_png}")
    print(f"Saved: {asv_png}")

    # Create comparison plots
    def plot_comparison(values_sym, values_asv, std_sym, std_asv, fname, title):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

        # Symmetric plot
        order = np.argsort(values_sym)[::-1]
        ax1.bar(np.array(feature_names)[order], values_sym[order],
                yerr=std_sym[order], capsize=5, alpha=0.7)
        ax1.set_xticklabels(np.array(feature_names)[order], rotation=45, ha='right')
        ax1.set_ylabel("Global Value")
        ax1.set_title(f"Symmetric (on-manifold)")
        ax1.grid(True, alpha=0.3)

        # ASV plot
        order = np.argsort(values_asv)[::-1]
        ax2.bar(np.array(feature_names)[order], values_asv[order],
                yerr=std_asv[order], capsize=5, alpha=0.7)
        ax2.set_xticklabels(np.array(feature_names)[order], rotation=45, ha='right')
        ax2.set_ylabel("Global Value")
        ax2.set_title(f"ASV (on-manifold)")
        ax2.grid(True, alpha=0.3)

        plt.suptitle(title)
        plt.tight_layout()
        plt.savefig(fname, dpi=150, bbox_inches='tight')
        plt.close()

    comparison_png = f"outputs/{out_prefix}_comparison_onmanifold.png"
    plot_comparison(global_sym, global_asv, global_sym_std, global_asv_std,
                   comparison_png, f"{out_prefix}: On-Manifold Comparison")
    print(f"Saved: {comparison_png}")

    # Console summary
    print(f"\nTop features (Symmetric on-manifold):")
    for i in np.argsort(global_sym)[::-1]:
        print(f"  {feature_names[i]:25s} {global_sym[i]: .6f} ± {global_sym_std[i]:.6f}")

    print(f"\nTop features (ASV on-manifold):")
    for i in np.argsort(global_asv)[::-1]:
        print(f"  {feature_names[i]:25s} {global_asv[i]: .6f} ± {global_asv_std[i]:.6f}")


def main():
    """Main execution function"""
    print("Perfect Paper-Faithful Asymmetric Shapley Values (ASV) Implementation")
    print("Based on arXiv:1910.06358v3 - Frye, Feige, & Rowat (2020)")
    print("=" * 80)

    # Define datasets to analyze
    tasks = [
        ("../../datasets/lung_cancer_risk.csv", "lung_cancer_risk",
         DATASET_CONSTRAINTS.get('lung_cancer_risk', {}), "lung_cancer_rf", "regression"),
        ("../../datasets/cv_risk.csv", "cv_risk",
         DATASET_CONSTRAINTS.get('cv_risk', {}), "cv_risk_rf", "regression"),
    ]

    for csv_path, target, constraints, prefix, task_type in tasks:
        if os.path.exists(csv_path):
            run_for_dataset(csv_path, target, constraints, prefix, task_type)
        else:
            print(f"Warning: {csv_path} not found, skipping...")

if __name__ == "__main__":
    main()

Perfect Paper-Faithful Asymmetric Shapley Values (ASV) Implementation
Based on arXiv:1910.06358v3 - Frye, Feige, & Rowat (2020)

=== DATASET: ../datasets/lung_cancer_risk.csv (target=lung_cancer_risk, task=regression) ===
Train R^2: 0.9643 | Test R^2: 0.7291
Causal DAG edges: [('smoking', 'stress'), ('smoking', 'drink_coffee')]
Evaluating on 50 test points...
  Processing point 0/50
  Processing point 10/50
  Processing point 20/50
  Processing point 30/50
  Processing point 40/50
Mean efficiency gap — Symmetric: 0.000000 | ASV: 0.000000
Saved: outputs/lung_cancer_rf_global_symmetric_onmanifold.csv
Saved: outputs/lung_cancer_rf_global_asv_onmanifold.csv
Saved: outputs/lung_cancer_rf_global_symmetric_onmanifold.png
Saved: outputs/lung_cancer_rf_global_asv_onmanifold.png


Saved: outputs/lung_cancer_rf_comparison_onmanifold.png

Top features (Symmetric on-manifold):
  drink_coffee              -0.178566 ± 1.775114
  stress                    -0.209363 ± 0.874311
  smoking                   -0.385857 ± 1.478393

Top features (ASV on-manifold):
  drink_coffee               0.055799 ± 0.964613
  stress                    -0.229514 ± 1.006356
  smoking                   -0.600071 ± 2.634941

=== DATASET: ../datasets/cv_risk.csv (target=cv_risk, task=regression) ===
Train R^2: 0.9540 | Test R^2: 0.6498
Causal DAG edges: [('family_history', 'bmi'), ('family_history', 'diet_score'), ('bmi', 'sleep_duration')]
Evaluating on 50 test points...
  Processing point 0/50
  Processing point 10/50
  Processing point 20/50
  Processing point 30/50
  Processing point 40/50
Mean efficiency gap — Symmetric: 0.000000 | ASV: 0.000000
Saved: outputs/cv_risk_rf_global_symmetric_onmanifold.csv
Saved: outputs/cv_risk_rf_global_asv_onmanifold.csv
Saved: outputs/cv_risk_rf_global_s

Saved: outputs/cv_risk_rf_comparison_onmanifold.png

Top features (Symmetric on-manifold):
  sleep_duration             0.188351 ± 0.677624
  family_history             0.032575 ± 0.266325
  diet_score                -0.092449 ± 0.709293
  bmi                       -0.162301 ± 1.794317

Top features (ASV on-manifold):
  family_history             0.121539 ± 0.598569
  sleep_duration             0.036871 ± 0.321621
  bmi                       -0.086897 ± 2.135618
  diet_score                -0.105337 ± 0.790482


In [6]:
#!/usr/bin/env python3

"""
Perfect Paper-Faithful Asymmetric Shapley Values (ASV) Implementation
Based on arXiv:1910.06358v3 - Frye, Feige, & Rowat (2020)

This implementation fixes all the issues in the original code to be truly paper-faithful:
1. Proper causal order filtering (distal constraint)
2. Correct conditional expectations (on-manifold)
3. Fair comparison (same imputer for both methods)
4. Global aggregation with efficiency checks
5. Proper topological sampling (not factorial enumeration)
6. Support for both regression and classification value functions
"""

import os
import random
from collections import defaultdict
from typing import Dict, List, Set, Tuple, Optional
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score


RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# Monte Carlo / sampling config
N_PERMUTATIONS_PER_POINT = 100   # how many permutations to sample per point
N_MC_CONDITIONAL = 50            # E[ f | x_S ] samples
N_MC_UNCONDITIONAL = 200         # E[ f ] samples for v(∅)
K_NEIGHBORS = 20                 # K for conditional KNN in S-subspace
TEST_SIZE = 0.25
MAX_TEST_POINTS = 50             # Limit for computational efficiency

# Causal constraints (distal ASV: ancestors must precede descendants)
DATASET_CONSTRAINTS: Dict[str, Dict[str, List[str]]] = {
    'cv_risk': {
        'family_history': ['bmi', 'diet_score'],
        'bmi': ['sleep_duration']
    },
    'lung_cancer_risk': {
        'smoking': ['stress', 'drink_coffee']
    },
}


def drop_unnamed_or_index_cols(df: pd.DataFrame) -> pd.DataFrame:
    """Remove unnamed or index-like columns"""
    drop_cols = []
    for c in df.columns:
        cl = str(c).strip().lower()
        if cl.startswith('unnamed') or cl in {'', 'index', 'id', 'x'}:
            drop_cols.append(c)
    if drop_cols:
        df = df.drop(columns=drop_cols)
    return df

def create_causal_dag(feature_names: List[str], edges: Dict[str, List[str]]) -> nx.DiGraph:
    """Create causal DAG from feature names and edges"""
    G = nx.DiGraph()
    G.add_nodes_from(feature_names)

    for cause, effects in edges.items():
        if cause in feature_names:
            for effect in effects:
                if effect in feature_names:
                    G.add_edge(cause, effect)

    if not nx.is_directed_acyclic_graph(G):
        raise ValueError("Causal relationships must form a DAG (no cycles)")

    return G

def sample_topological_orders(G: nx.DiGraph, n_samples: int) -> List[List[str]]:
    """
    Sample n_samples random topological orders using Kahn's algorithm with random tie-breaking
    This is more efficient than enumerating all permutations
    """
    orders = []

    for _ in range(n_samples):
        # Kahn's algorithm with random tie-breaking
        in_degree = {node: G.in_degree(node) for node in G.nodes()}
        queue = [node for node in G.nodes() if in_degree[node] == 0]
        order = []

        while queue:
            # Random tie-breaking
            random.shuffle(queue)
            node = queue.pop(0)
            order.append(node)

            # Update in-degrees of neighbors
            for neighbor in G.neighbors(node):
                in_degree[neighbor] -= 1
                if in_degree[neighbor] == 0:
                    queue.append(neighbor)

        orders.append(order)

    return orders

def efficiency_gap_local(phi: np.ndarray, vN: float, v0: float) -> float:
    """Check efficiency property: sum(phi) should equal v(N) - v(∅)"""
    return abs(phi.sum() - (vN - v0))


class ConditionalKNNImputer:
    """
    Conditional splicing sampler for on-manifold v_x(S).
    This implements the paper's on-manifold value function using KNN-conditional sampling.
    """
    def __init__(self, X_train: np.ndarray, feature_names: List[str]):
        self.X_train = np.asarray(X_train)
        self.feature_names = feature_names
        self.n, self.d = self.X_train.shape
        self.scaler = StandardScaler().fit(self.X_train)
        self.Xs = self.scaler.transform(self.X_train)

    def sample_conditional(self, x: np.ndarray, S: Set[int], n_mc: int) -> np.ndarray:
        """
        Sample from p(x'_{~S} | x_S) using KNN-conditional sampling
        This implements the paper's on-manifold conditional expectation
        """
        x = np.asarray(x)
        if len(S) == 0:
            # Unconditional: sample from training data
            idx = np.random.randint(0, self.n, size=n_mc)
            return self.X_train[idx].copy()

        # Find k-nearest neighbors in the S-subspace
        cols = sorted(list(S))
        Xs_S = self.Xs[:, cols]
        xS = self.scaler.transform(x.reshape(1, -1))[0, cols].reshape(1, -1)

        k = min(K_NEIGHBORS, self.n)
        nbrs = NearestNeighbors(n_neighbors=k, algorithm='auto', metric='euclidean')
        nbrs.fit(Xs_S)
        _, neigh_idx = nbrs.kneighbors(xS, return_distance=True)
        neigh_idx = neigh_idx[0]

        # Sample from neighbors and condition on S
        comp = [j for j in range(self.d) if j not in S]
        out = np.tile(x, (n_mc, 1))
        pick = np.random.choice(neigh_idx, size=n_mc, replace=True)
        out[:, comp] = self.X_train[pick][:, comp]
        return out


class OnManifoldValueFunction:
    """
    On-manifold value function v_x(S) = E_{x'|x_S}[f(x_S ⊕ x'_{~S})]
    This implements Equation (4) from the paper exactly
    """
    def __init__(self, model, imputer: ConditionalKNNImputer, task_type: str = 'regression'):
        self.model = model
        self.imputer = imputer
        self.task_type = task_type
        self.cache = {}

    def _cache_key(self, x: np.ndarray, S: Set[int], n_mc: int) -> str:
        """Create cache key for memoization"""
        return f"{hash(x.tobytes())}_{tuple(sorted(S))}_{n_mc}"

    def v_empty(self, n_mc: int = N_MC_UNCONDITIONAL) -> float:
        """v(∅) = E_{x'}[f(x')] - baseline expectation"""
        key = self._cache_key(np.zeros(self.imputer.d), set(), n_mc)
        if key in self.cache:
            return self.cache[key]

        xs = self.imputer.sample_conditional(x=np.zeros(self.imputer.d), S=set(), n_mc=n_mc)

        if self.task_type == 'regression':
            pred = float(self.model.predict(xs).mean())
        else:  # classification
            pred = float(self.model.predict_proba(xs).mean())

        self.cache[key] = pred
        return pred

    def v(self, x: np.ndarray, S: Set[int], n_mc: int = N_MC_CONDITIONAL) -> float:
        """v(S) = E_{x'|x_S}[f(x_S ⊕ x'_{~S})] - conditional expectation"""
        key = self._cache_key(x, S, n_mc)
        if key in self.cache:
            return self.cache[key]

        xs = self.imputer.sample_conditional(x, S, n_mc=n_mc)

        if self.task_type == 'regression':
            pred = float(self.model.predict(xs).mean())
        else:  # classification
            pred = float(self.model.predict_proba(xs).mean())

        self.cache[key] = pred
        return pred


def local_asv_for_point(x: np.ndarray,
                        vfun: OnManifoldValueFunction,
                        feature_names: List[str],
                        G: nx.DiGraph,
                        n_perms: int = N_PERMUTATIONS_PER_POINT) -> Tuple[np.ndarray, float, float]:
    """
    Local ASV using order-restricted permutations (distal constraint)
    This implements Equation (10) from the paper exactly
    """
    d = len(feature_names)
    name_to_idx = {f: i for i, f in enumerate(feature_names)}

    # Sample topological orders consistent with causal DAG
    perms_names = sample_topological_orders(G, n_perms)
    perms_idx = [[name_to_idx[nm] for nm in perm] for perm in perms_names]

    v0 = vfun.v_empty()
    vN = vfun.v(x, set(range(d)))
    phi = np.zeros(d, dtype=float)

    # For each valid permutation π
    for perm in perms_idx:
        S = set()
        prev = v0
        for j in perm:
            curr = vfun.v(x, S | {j})
            phi[j] += (curr - prev)  # Marginal contribution
            prev = curr
            S.add(j)

    phi /= len(perms_idx)  # Average over all valid permutations
    return phi, vN, v0

def local_symmetric_shap_for_point(x: np.ndarray,
                                   vfun: OnManifoldValueFunction,
                                   d: int,
                                   n_perms: int = N_PERMUTATIONS_PER_POINT) -> Tuple[np.ndarray, float, float]:
    """
    Local symmetric SHAP using uniform random permutations with the SAME on-manifold v
    This ensures fair comparison with ASV
    """
    v0 = vfun.v_empty()
    vN = vfun.v(x, set(range(d)))
    phi = np.zeros(d, dtype=float)

    for _ in range(n_perms):
        perm = list(range(d))
        random.shuffle(perm)  # Uniform random permutation
        S = set()
        prev = v0
        for j in perm:
            curr = vfun.v(x, S | {j})
            phi[j] += (curr - prev)  # Marginal contribution
            prev = curr
            S.add(j)

    phi /= n_perms  # Average over all permutations
    return phi, vN, v0


def train_model(X_train, y_train, task_type: str = 'regression'):
    """Train Random Forest model based on task type"""
    if task_type == 'regression':
        model = RandomForestRegressor(
            n_estimators=200,
            max_features="sqrt",
            random_state=RANDOM_SEED,
            n_jobs=-1
        )
    else:  # classification
        model = RandomForestClassifier(
            n_estimators=200,
            max_features="sqrt",
            random_state=RANDOM_SEED,
            n_jobs=-1
        )

    model.fit(X_train, y_train)
    return model

def run_for_dataset(csv_path: str,
                    target_col: str,
                    constraints: Dict[str, List[str]],
                    out_prefix: str,
                    task_type: str = 'regression'):
    """Run analysis for a single dataset"""
    print(f"\n=== DATASET: {csv_path} (target={target_col}, task={task_type}) ===")

    # Load and prepare data
    df = pd.read_csv(csv_path)
    df = drop_unnamed_or_index_cols(df)

    assert target_col in df.columns, f"Target '{target_col}' not found in {csv_path}"
    y = pd.to_numeric(df[target_col], errors='coerce').values.astype(float)
    X_df = df.drop(columns=[target_col]).apply(pd.to_numeric, errors='coerce')
    feature_names = list(X_df.columns)
    X = X_df.values.astype(float)

    # Basic numeric imputation
    X = SimpleImputer(strategy='median').fit_transform(X)

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED
    )

    # Train model
    model = train_model(X_train, y_train, task_type)

    if task_type == 'regression':
        train_score = model.score(X_train, y_train)
        test_score = model.score(X_test, y_test)
        print(f"Train R^2: {train_score:.4f} | Test R^2: {test_score:.4f}")
    else:
        train_score = model.score(X_train, y_train)
        test_score = model.score(X_test, y_test)
        print(f"Train Accuracy: {train_score:.4f} | Test Accuracy: {test_score:.4f}")

    # Create causal DAG
    G = create_causal_dag(feature_names, constraints)
    print(f"Causal DAG edges: {list(G.edges())}")

    # On-manifold value function (same imputer for both methods)
    imputer = ConditionalKNNImputer(X_train, feature_names)
    vfun = OnManifoldValueFunction(model, imputer, task_type)
    d = X.shape[1]

    # Evaluate on test points
    n_eval = min(MAX_TEST_POINTS, len(X_test))
    if len(X_test) > n_eval:
        idx = np.random.choice(len(X_test), size=n_eval, replace=False)
        X_eval = X_test[idx]
    else:
        X_eval = X_test

    print(f"Evaluating on {len(X_eval)} test points...")

    # Compute local explanations
    sym_vals = []
    asv_vals = []
    eff_gaps_sym = []
    eff_gaps_asv = []

    for i, x in enumerate(X_eval):
        if i % 10 == 0:
            print(f"  Processing point {i}/{len(X_eval)}")

        # Symmetric SHAP (on-manifold)
        phi_sym, vN, v0 = local_symmetric_shap_for_point(x, vfun, d)
        sym_vals.append(phi_sym)
        eff_gaps_sym.append(efficiency_gap_local(phi_sym, vN, v0))

        # ASV (distal) with constraints
        phi_asv, vN2, v0_2 = local_asv_for_point(x, vfun, feature_names, G)
        asv_vals.append(phi_asv)
        eff_gaps_asv.append(efficiency_gap_local(phi_asv, vN2, v0_2))

    # Aggregate to global values
    sym_vals = np.vstack(sym_vals)
    asv_vals = np.vstack(asv_vals)
    global_sym = sym_vals.mean(axis=0)
    global_asv = asv_vals.mean(axis=0)
    global_sym_std = sym_vals.std(axis=0)
    global_asv_std = asv_vals.std(axis=0)

    print(f"Mean efficiency gap — Symmetric: {np.mean(eff_gaps_sym):.6f} | ASV: {np.mean(eff_gaps_asv):.6f}")

    # Save outputs
    os.makedirs("outputs", exist_ok=True)

    # Save results
    df_sym = pd.DataFrame({
        'feature': feature_names,
        'global_value': global_sym,
        'std': global_sym_std
    })
    df_asv = pd.DataFrame({
        'feature': feature_names,
        'global_value': global_asv,
        'std': global_asv_std
    })

    sym_csv = f"outputs/{out_prefix}_global_symmetric_onmanifold.csv"
    asv_csv = f"outputs/{out_prefix}_global_asv_onmanifold.csv"
    df_sym.to_csv(sym_csv, index=False)
    df_asv.to_csv(asv_csv, index=False)
    print(f"Saved: {sym_csv}")
    print(f"Saved: {asv_csv}")

    def plot_single_bar(values, std, fname, title):
        order = np.argsort(values)[::-1]
        plt.figure(figsize=(9, 5))
        plt.bar(np.array(feature_names)[order], values[order], yerr=std[order], capsize=5, alpha=0.8)
        plt.xticks(rotation=60, ha='right')
        plt.ylabel("Global Value")
        plt.title(title)
        plt.tight_layout()
        plt.savefig(fname, dpi=150)
        plt.close()

    sym_png = f"outputs/{out_prefix}_global_symmetric_onmanifold.png"
    asv_png = f"outputs/{out_prefix}_global_asv_onmanifold.png"
    plot_single_bar(global_sym, global_sym_std, sym_png, f"{out_prefix}: Global Symmetric (on-manifold)")
    plot_single_bar(global_asv, global_asv_std, asv_png, f"{out_prefix}: Global ASV (on-manifold)")
    print(f"Saved: {sym_png}")
    print(f"Saved: {asv_png}")

    def plot_like_example(values, feature_names, fname, dataset_name, subtitle="Bar: first column"):
        plt.figure(figsize=(9, 5))
        # keep original feature order (do NOT sort)
        plt.bar(feature_names, values)
        plt.xticks(rotation=45, ha='right')
        plt.ylabel("Attribution")
        plt.title(f"Feature Attribution Comparison — {dataset_name}\n({subtitle})")
        plt.tight_layout()
        plt.savefig(fname, dpi=150)
        plt.close()

    bar_sym_png = f"outputs/{out_prefix}_bar_symmetric_like_example.png"
    bar_asv_png = f"outputs/{out_prefix}_bar_asv_like_example.png"
    plot_like_example(global_sym, feature_names, bar_sym_png, out_prefix, "Bar: global symmetric")
    plot_like_example(global_asv, feature_names, bar_asv_png, out_prefix, "Bar: global ASV")
    print(f"Saved: {bar_sym_png}")
    print(f"Saved: {bar_asv_png}")

    bar_local_sym_png = f"outputs/{out_prefix}_bar_local0_symmetric_like_example.png"
    bar_local_asv_png = f"outputs/{out_prefix}_bar_local0_asv_like_example.png"
    plot_like_example(sym_vals[0], feature_names, bar_local_sym_png, out_prefix, "Bar: first column (local, symmetric)")
    plot_like_example(asv_vals[0], feature_names, bar_local_asv_png, out_prefix, "Bar: first column (local, ASV)")
    print(f"Saved: {bar_local_sym_png}")
    print(f"Saved: {bar_local_asv_png}")

    # Create comparison plots (existing)
    def plot_comparison(values_sym, values_asv, std_sym, std_asv, fname, title):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

        # Symmetric plot
        order = np.argsort(values_sym)[::-1]
        ax1.bar(np.array(feature_names)[order], values_sym[order],
                yerr=std_sym[order], capsize=5, alpha=0.7)
        ax1.set_xticklabels(np.array(feature_names)[order], rotation=45, ha='right')
        ax1.set_ylabel("Global Value")
        ax1.set_title(f"Symmetric (on-manifold)")
        ax1.grid(True, alpha=0.3)

        # ASV plot
        order = np.argsort(values_asv)[::-1]
        ax2.bar(np.array(feature_names)[order], values_asv[order],
                yerr=std_asv[order], capsize=5, alpha=0.7)
        ax2.set_xticklabels(np.array(feature_names)[order], rotation=45, ha='right')
        ax2.set_ylabel("Global Value")
        ax2.set_title(f"ASV (on-manifold)")
        ax2.grid(True, alpha=0.3)

        plt.suptitle(title)
        plt.tight_layout()
        plt.savefig(fname, dpi=150, bbox_inches='tight')
        plt.close()

    comparison_png = f"outputs/{out_prefix}_comparison_onmanifold.png"
    plot_comparison(global_sym, global_asv, global_sym_std, global_asv_std,
                    comparison_png, f"{out_prefix}: On-Manifold Comparison")
    print(f"Saved: {comparison_png}")

    # Console summary
    print(f"\nTop features (Symmetric on-manifold):")
    for i in np.argsort(global_sym)[::-1]:
        print(f"  {feature_names[i]:25s} {global_sym[i]: .6f} ± {global_sym_std[i]:.6f}")

    print(f"\nTop features (ASV on-manifold):")
    for i in np.argsort(global_asv)[::-1]:
        print(f"  {feature_names[i]:25s} {global_asv[i]: .6f} ± {global_asv_std[i]:.6f}")


def main():
    """Main execution function"""
    print("Perfect Paper-Faithful Asymmetric Shapley Values (ASV) Implementation")
    print("Based on arXiv:1910.06358v3 - Frye, Feige, & Rowat (2020)")
    print("=" * 80)

    # Define datasets to analyze
    tasks = [
        ("../../datasets/lung_cancer_risk.csv", "lung_cancer_risk",
         DATASET_CONSTRAINTS.get('lung_cancer_risk', {}), "lung_cancer_rf", "regression"),
        ("../../datasets/cv_risk.csv", "cv_risk",
         DATASET_CONSTRAINTS.get('cv_risk', {}), "cv_risk_rf", "regression"),
    ]

    for csv_path, target, constraints, prefix, task_type in tasks:
        if os.path.exists(csv_path):
            run_for_dataset(csv_path, target, constraints, prefix, task_type)
        else:
            print(f"Warning: {csv_path} not found, skipping...")

if __name__ == "__main__":
    main()

Perfect Paper-Faithful Asymmetric Shapley Values (ASV) Implementation
Based on arXiv:1910.06358v3 - Frye, Feige, & Rowat (2020)

=== DATASET: ../datasets/lung_cancer_risk.csv (target=lung_cancer_risk, task=regression) ===
Train R^2: 0.9643 | Test R^2: 0.7291
Causal DAG edges: [('smoking', 'stress'), ('smoking', 'drink_coffee')]
Evaluating on 50 test points...
  Processing point 0/50
  Processing point 10/50
  Processing point 20/50
  Processing point 30/50
  Processing point 40/50
Mean efficiency gap — Symmetric: 0.000000 | ASV: 0.000000
Saved: outputs/lung_cancer_rf_global_symmetric_onmanifold.csv
Saved: outputs/lung_cancer_rf_global_asv_onmanifold.csv
Saved: outputs/lung_cancer_rf_global_symmetric_onmanifold.png
Saved: outputs/lung_cancer_rf_global_asv_onmanifold.png
Saved: outputs/lung_cancer_rf_bar_symmetric_like_example.png
Saved: outputs/lung_cancer_rf_bar_asv_like_example.png
Saved: outputs/lung_cancer_rf_bar_local0_symmetric_like_example.png
Saved: outputs/lung_cancer_rf_bar_lo

Saved: outputs/lung_cancer_rf_comparison_onmanifold.png

Top features (Symmetric on-manifold):
  drink_coffee              -0.178566 ± 1.775114
  stress                    -0.209363 ± 0.874311
  smoking                   -0.385857 ± 1.478393

Top features (ASV on-manifold):
  drink_coffee               0.055799 ± 0.964613
  stress                    -0.229514 ± 1.006356
  smoking                   -0.600071 ± 2.634941

=== DATASET: ../datasets/cv_risk.csv (target=cv_risk, task=regression) ===
Train R^2: 0.9540 | Test R^2: 0.6498
Causal DAG edges: [('family_history', 'bmi'), ('family_history', 'diet_score'), ('bmi', 'sleep_duration')]
Evaluating on 50 test points...
  Processing point 0/50
  Processing point 10/50
  Processing point 20/50
  Processing point 30/50
  Processing point 40/50
Mean efficiency gap — Symmetric: 0.000000 | ASV: 0.000000
Saved: outputs/cv_risk_rf_global_symmetric_onmanifold.csv
Saved: outputs/cv_risk_rf_global_asv_onmanifold.csv
Saved: outputs/cv_risk_rf_global_s

Saved: outputs/cv_risk_rf_comparison_onmanifold.png

Top features (Symmetric on-manifold):
  sleep_duration             0.188351 ± 0.677624
  family_history             0.032575 ± 0.266325
  diet_score                -0.092449 ± 0.709293
  bmi                       -0.162301 ± 1.794317

Top features (ASV on-manifold):
  family_history             0.121539 ± 0.598569
  sleep_duration             0.036871 ± 0.321621
  bmi                       -0.086897 ± 2.135618
  diet_score                -0.105337 ± 0.790482


In [10]:
#!/usr/bin/env python3

"""
Complete Paper-Faithful Asymmetric Shapley Values (ASV) Implementation
Based on arXiv:1910.06358v3 - Frye, Feige, & Rowat (2020)

All fixes applied:
1. Proper causal DAG with cycle detection
2. Correct classification value function with true labels
3. Secure cache key hashing
4. Fair comparison (same imputer for both methods)
5. Global aggregation matching paper's Equation (5)
6. Proper topological sampling with Kahn's algorithm
7. Support for both regression and classification
"""

import os
import random
import hashlib
from collections import defaultdict
from typing import Dict, List, Set, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer


RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# Monte Carlo / sampling config
N_PERMUTATIONS_PER_POINT = 100   # Permutations per point (paper uses more)
N_MC_CONDITIONAL = 50            # E[f | x_S] samples
N_MC_UNCONDITIONAL = 200         # E[f] samples for v(∅)
K_NEIGHBORS = 20                 # K for conditional KNN
TEST_SIZE = 0.25
MAX_TEST_POINTS = 50             # Limit for computational efficiency

# Causal constraints (distal ASV: ancestors must precede descendants)
DATASET_CONSTRAINTS: Dict[str, Dict[str, List[str]]] = {
    'cv_risk': {
        'family_history': ['bmi', 'diet_score'],
        'age': ['bmi', 'cholesterol'],
        'bmi': ['sleep_duration']
    },
    'lung_cancer_risk': {
        'smoking': ['stress', 'drink_coffee', 'lung_cancer_risk'],
        'age': ['smoking'],
        'family_history': ['lung_cancer_risk']
    },
}


def drop_unnamed_or_index_cols(df: pd.DataFrame) -> pd.DataFrame:
    """Remove unnamed or index-like columns"""
    drop_cols = []
    for c in df.columns:
        cl = str(c).strip().lower()
        if cl.startswith('unnamed') or cl in {'', 'index', 'id', 'x'}:
            drop_cols.append(c)
    if drop_cols:
        df = df.drop(columns=drop_cols)
    return df

def create_causal_dag(feature_names: List[str], edges: Dict[str, List[str]]) -> nx.DiGraph:
    """
    Create causal DAG from feature names and edges
    Validates that the graph is acyclic (required for causal interpretation)
    """
    G = nx.DiGraph()
    G.add_nodes_from(feature_names)

    for cause, effects in edges.items():
        if cause in feature_names:
            for effect in effects:
                if effect in feature_names:
                    G.add_edge(cause, effect)

    if not nx.is_directed_acyclic_graph(G):
        raise ValueError("Causal relationships must form a DAG (no cycles)")

    return G

def sample_topological_orders(G: nx.DiGraph, n_samples: int) -> List[List[str]]:
    """
    Sample n_samples random topological orders using Kahn's algorithm with random tie-breaking
    This implements the paper's distal approach (Equation 11)
    """
    orders = []

    for _ in range(n_samples):
        # Kahn's algorithm with random tie-breaking
        in_degree = {node: G.in_degree(node) for node in G.nodes()}
        queue = [node for node in G.nodes() if in_degree[node] == 0]
        order = []

        while queue:
            # Random tie-breaking ensures uniform sampling over valid orders
            random.shuffle(queue)
            node = queue.pop(0)
            order.append(node)

            # Update in-degrees of neighbors
            for neighbor in G.neighbors(node):
                in_degree[neighbor] -= 1
                if in_degree[neighbor] == 0:
                    queue.append(neighbor)

        if len(order) != len(G.nodes()):
            raise ValueError("Graph has cycles - cannot produce topological order")

        orders.append(order)

    return orders

def efficiency_gap_local(phi: np.ndarray, vN: float, v0: float) -> float:
    """
    Check Axiom 1 (Efficiency): sum(phi) should equal v(N) - v(∅)
    This is a key property that both Shapley and ASV must satisfy
    """
    return abs(phi.sum() - (vN - v0))


class ConditionalKNNImputer:
    """
    Conditional splicing sampler for on-manifold v_x(S).
    Implements the paper's on-manifold value function (Equation 4) using KNN.

    Note: Paper uses VAE-based imputation (Appendix B), but KNN is simpler
    and still respects the conditional distribution structure.
    """
    def __init__(self, X_train: np.ndarray, feature_names: List[str]):
        self.X_train = np.asarray(X_train)
        self.feature_names = feature_names
        self.n, self.d = self.X_train.shape

        # Standardize for distance-based KNN
        self.scaler = StandardScaler().fit(self.X_train)
        self.Xs = self.scaler.transform(self.X_train)

    def sample_conditional(self, x: np.ndarray, S: Set[int], n_mc: int) -> np.ndarray:
        """
        Sample from p(x'_{~S} | x_S) using KNN-conditional sampling

        This implements: x_S ⊕ x'_{~S} where x' ~ p(x'|x_S)
        """
        x = np.asarray(x)

        if len(S) == 0:
            # Unconditional: sample from training data (baseline)
            idx = np.random.randint(0, self.n, size=n_mc)
            return self.X_train[idx].copy()

        # Find k-nearest neighbors in the S-subspace
        cols = sorted(list(S))
        Xs_S = self.Xs[:, cols]
        xS = self.scaler.transform(x.reshape(1, -1))[0, cols].reshape(1, -1)

        k = min(K_NEIGHBORS, self.n)
        nbrs = NearestNeighbors(n_neighbors=k, algorithm='auto', metric='euclidean')
        nbrs.fit(Xs_S)
        _, neigh_idx = nbrs.kneighbors(xS, return_distance=True)
        neigh_idx = neigh_idx[0]

        # Sample from neighbors and splice: x_S from x, x_{~S} from neighbors
        comp = [j for j in range(self.d) if j not in S]
        out = np.tile(x, (n_mc, 1))
        pick = np.random.choice(neigh_idx, size=n_mc, replace=True)
        out[:, comp] = self.X_train[pick][:, comp]

        return out


class OnManifoldValueFunction:
    """
    On-manifold value function implementing Equation (4) from the paper:

    v_x(S) = E_{x'|x_S}[f(x_S ⊕ x'_{~S})]

    For regression: f(x) is the predicted value
    For classification: f_y(x) is the probability assigned to true class y
    """
    def __init__(self, model, imputer: ConditionalKNNImputer,
                 task_type: str = 'regression', y_true: Optional[int] = None):
        self.model = model
        self.imputer = imputer
        self.task_type = task_type
        self.y_true = y_true  # For classification: true label of the point
        self.cache = {}

    def _cache_key(self, x: np.ndarray, S: Set[int], n_mc: int) -> str:
        """Create secure cache key using SHA256 hash"""
        x_hash = hashlib.sha256(x.tobytes()).hexdigest()[:16]
        return f"{x_hash}_{tuple(sorted(S))}_{n_mc}"

    def v_empty(self, n_mc: int = N_MC_UNCONDITIONAL) -> float:
        """
        v(∅) = E_{x'}[f(x')] - baseline expectation

        For classification: E[f_y(x')] where we average over y too
        """
        key = f"empty_{n_mc}_{self.task_type}"
        if key in self.cache:
            return self.cache[key]

        xs = self.imputer.sample_conditional(
            x=np.zeros(self.imputer.d), S=set(), n_mc=n_mc
        )

        if self.task_type == 'regression':
            pred = float(self.model.predict(xs).mean())
        else:  # classification
            if self.y_true is not None:
                # For a specific point: probability of its true class
                proba = self.model.predict_proba(xs)
                pred = float(proba[:, self.y_true].mean())
            else:
                # For baseline: average probability across all classes
                pred = float(self.model.predict_proba(xs).mean())

        self.cache[key] = pred
        return pred

    def v(self, x: np.ndarray, S: Set[int], n_mc: int = N_MC_CONDITIONAL) -> float:
        """
        v(S) = E_{x'|x_S}[f(x_S ⊕ x'_{~S})] - conditional expectation

        This is the core of the on-manifold approach (Equation 4)
        """
        key = self._cache_key(x, S, n_mc)
        if key in self.cache:
            return self.cache[key]

        xs = self.imputer.sample_conditional(x, S, n_mc=n_mc)

        if self.task_type == 'regression':
            pred = float(self.model.predict(xs).mean())
        else:  # classification
            # Paper Eq (2): f_y(x) is prob of TRUE class y for point x
            proba = self.model.predict_proba(xs)
            pred = float(proba[:, self.y_true].mean())

        self.cache[key] = pred
        return pred


def local_asv_for_point(x: np.ndarray,
                        vfun: OnManifoldValueFunction,
                        feature_names: List[str],
                        G: nx.DiGraph,
                        n_perms: int = N_PERMUTATIONS_PER_POINT) -> Tuple[np.ndarray, float, float]:
    """
    Local ASV using causal order-restricted permutations (distal approach)

    This implements Equation (10) from the paper:
    φ^(w)_v(i) = Σ_π w(π)[v({j: π(j) ≤ π(i)}) - v({j: π(j) < π(i)})]

    where w(π) places weight only on causal-order-respecting permutations
    """
    d = len(feature_names)
    name_to_idx = {f: i for i, f in enumerate(feature_names)}

    # Sample topological orders consistent with causal DAG
    perms_names = sample_topological_orders(G, n_perms)
    perms_idx = [[name_to_idx[nm] for nm in perm] for perm in perms_names]

    v0 = vfun.v_empty()
    vN = vfun.v(x, set(range(d)))
    phi = np.zeros(d, dtype=float)

    # For each valid permutation π
    for perm in perms_idx:
        S = set()
        prev = v0
        for j in perm:
            curr = vfun.v(x, S | {j})
            phi[j] += (curr - prev)  # Marginal contribution
            prev = curr
            S.add(j)

    phi /= len(perms_idx)  # Average over all valid permutations
    return phi, vN, v0

def local_symmetric_shap_for_point(x: np.ndarray,
                                   vfun: OnManifoldValueFunction,
                                   d: int,
                                   n_perms: int = N_PERMUTATIONS_PER_POINT) -> Tuple[np.ndarray, float, float]:
    """
    Local symmetric SHAP using uniform random permutations

    This implements standard Shapley values (Equation 1) with the same
    on-manifold value function, ensuring fair comparison with ASV.
    """
    v0 = vfun.v_empty()
    vN = vfun.v(x, set(range(d)))
    phi = np.zeros(d, dtype=float)

    for _ in range(n_perms):
        perm = list(range(d))
        random.shuffle(perm)  # Uniform random permutation (Axiom 4)
        S = set()
        prev = v0
        for j in perm:
            curr = vfun.v(x, S | {j})
            phi[j] += (curr - prev)  # Marginal contribution
            prev = curr
            S.add(j)

    phi /= n_perms  # Average over all permutations
    return phi, vN, v0


def train_model(X_train, y_train, task_type: str = 'regression'):
    """Train Random Forest model based on task type"""
    if task_type == 'regression':
        model = RandomForestRegressor(
            n_estimators=200,
            max_features="sqrt",
            random_state=RANDOM_SEED,
            n_jobs=-1
        )
    else:  # classification
        model = RandomForestClassifier(
            n_estimators=200,
            max_features="sqrt",
            random_state=RANDOM_SEED,
            n_jobs=-1
        )

    model.fit(X_train, y_train)
    return model

def run_for_dataset(csv_path: str,
                    target_col: str,
                    constraints: Dict[str, List[str]],
                    out_prefix: str,
                    task_type: str = 'regression'):
    """
    Run complete ASV analysis for a single dataset

    Implements the paper's methodology:
    1. Train model
    2. Create causal DAG from constraints
    3. Compute local explanations (Symmetric + ASV)
    4. Aggregate to global values (Equation 5)
    5. Verify efficiency axiom
    """
    print(f"\n{'='*80}")
    print(f"DATASET: {csv_path}")
    print(f"Target: {target_col} | Task: {task_type}")
    print(f"{'='*80}")

    # Load and prepare data
    df = pd.read_csv(csv_path)
    df = drop_unnamed_or_index_cols(df)

    assert target_col in df.columns, f"Target '{target_col}' not found"

    # Prepare target
    if task_type == 'regression':
        y = pd.to_numeric(df[target_col], errors='coerce').values.astype(float)
    else:
        # For classification, ensure labels are integers
        y = pd.to_numeric(df[target_col], errors='coerce').values.astype(int)

    # Prepare features
    X_df = df.drop(columns=[target_col]).apply(pd.to_numeric, errors='coerce')
    feature_names = list(X_df.columns)
    X = X_df.values.astype(float)

    # Basic numeric imputation
    imputer_basic = SimpleImputer(strategy='median')
    X = imputer_basic.fit_transform(X)

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y if task_type == 'classification' else None
    )

    # Train model
    print("\nTraining model...")
    model = train_model(X_train, y_train, task_type)

    train_score = model.score(X_train, y_train)
    test_score = model.score(X_test, y_test)

    if task_type == 'regression':
        print(f"Train R²: {train_score:.4f} | Test R²: {test_score:.4f}")
    else:
        print(f"Train Accuracy: {train_score:.4f} | Test Accuracy: {test_score:.4f}")

    # Create causal DAG
    print("\nCreating causal DAG...")
    G = create_causal_dag(feature_names, constraints)
    print(f"Causal edges: {list(G.edges())}")
    if len(G.edges()) == 0:
        print("WARNING: No causal constraints - ASV will equal Symmetric Shapley!")

    # Create shared imputer (ensures fair comparison)
    print("\nInitializing on-manifold imputer...")
    imputer_conditional = ConditionalKNNImputer(X_train, feature_names)
    d = X.shape[1]

    # Subsample test points for efficiency
    n_eval = min(MAX_TEST_POINTS, len(X_test))
    if len(X_test) > n_eval:
        idx = np.random.choice(len(X_test), size=n_eval, replace=False)
        X_eval = X_test[idx]
        y_eval = y_test[idx]
    else:
        X_eval = X_test
        y_eval = y_test

    print(f"\nEvaluating on {len(X_eval)} test points...")

    # Compute local explanations for each test point
    sym_vals = []
    asv_vals = []
    eff_gaps_sym = []
    eff_gaps_asv = []

    for i, (x, y_true) in enumerate(zip(X_eval, y_eval)):
        if i % 10 == 0:
            print(f"  Processing point {i+1}/{len(X_eval)}")

        # Create value function for this specific point
        # For classification: tracks true label y for f_y(x)
        vfun = OnManifoldValueFunction(
            model, imputer_conditional, task_type,
            y_true=int(y_true) if task_type == 'classification' else None
        )

        # Symmetric SHAP (on-manifold, uniform permutations)
        phi_sym, vN_sym, v0_sym = local_symmetric_shap_for_point(x, vfun, d)
        sym_vals.append(phi_sym)
        eff_gaps_sym.append(efficiency_gap_local(phi_sym, vN_sym, v0_sym))

        # ASV (on-manifold, causal-order permutations)
        phi_asv, vN_asv, v0_asv = local_asv_for_point(x, vfun, feature_names, G)
        asv_vals.append(phi_asv)
        eff_gaps_asv.append(efficiency_gap_local(phi_asv, vN_asv, v0_asv))

    # Aggregate to global values (Equation 5 from paper)
    print("\nAggregating to global values...")
    sym_vals = np.vstack(sym_vals)
    asv_vals = np.vstack(asv_vals)

    global_sym = sym_vals.mean(axis=0)
    global_asv = asv_vals.mean(axis=0)
    global_sym_std = sym_vals.std(axis=0)
    global_asv_std = asv_vals.std(axis=0)

    print(f"\nEfficiency verification:")
    print(f"  Mean gap (Symmetric): {np.mean(eff_gaps_sym):.6f}")
    print(f"  Mean gap (ASV):       {np.mean(eff_gaps_asv):.6f}")
    print(f"  (Should be close to 0 - verifies Axiom 1)")

    # Save outputs
    os.makedirs("outputs", exist_ok=True)

    # Save CSV results
    df_sym = pd.DataFrame({
        'feature': feature_names,
        'global_value': global_sym,
        'std': global_sym_std
    })
    df_asv = pd.DataFrame({
        'feature': feature_names,
        'global_value': global_asv,
        'std': global_asv_std
    })

    sym_csv = f"outputs/{out_prefix}_global_symmetric_onmanifold.csv"
    asv_csv = f"outputs/{out_prefix}_global_asv_onmanifold.csv"
    df_sym.to_csv(sym_csv, index=False)
    df_asv.to_csv(asv_csv, index=False)
    print(f"\nSaved: {sym_csv}")
    print(f"Saved: {asv_csv}")

    def plot_comparison(values_sym, values_asv, std_sym, std_asv, fname, title):
        """Create side-by-side comparison plots"""
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

        # Symmetric plot
        order = np.argsort(values_sym)[::-1]
        ax1.bar(range(len(feature_names)), values_sym[order],
                yerr=std_sym[order], capsize=5, alpha=0.85, color='steelblue',
                edgecolor='navy', linewidth=0.5)
        ax1.set_xticks(range(len(feature_names)))
        ax1.set_xticklabels(np.array(feature_names)[order], rotation=45, ha='right')
        ax1.set_ylabel("Global Value", fontsize=12)
        ax1.set_title("Symmetric Shapley (on-manifold)", fontsize=14, fontweight='bold')
        ax1.grid(True, alpha=0.3, axis='y')
        ax1.axhline(y=0, color='k', linestyle='-', linewidth=0.5)

        # ASV plot
        order = np.argsort(values_asv)[::-1]
        ax2.bar(range(len(feature_names)), values_asv[order],
                yerr=std_asv[order], capsize=5, alpha=0.85, color='coral',
                edgecolor='darkred', linewidth=0.5)
        ax2.set_xticks(range(len(feature_names)))
        ax2.set_xticklabels(np.array(feature_names)[order], rotation=45, ha='right')
        ax2.set_ylabel("Global Value", fontsize=12)
        ax2.set_title("ASV (on-manifold, causal)", fontsize=14, fontweight='bold')
        ax2.grid(True, alpha=0.3, axis='y')
        ax2.axhline(y=0, color='k', linestyle='-', linewidth=0.5)

        plt.suptitle(title, fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.savefig(fname, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"Saved: {fname}")

    def plot_combined_comparison(values_sym, values_asv, fname_sym, fname_asv, title_base):
        """Create individual plots matching your exact style - one bar chart per method"""

        # Plot 1: Symmetric Shapley
        fig, ax = plt.subplots(figsize=(10, 6))
        order = np.argsort(values_sym)[::-1]
        colors = ['#1f77b4' if v >= 0 else '#d62728' for v in values_sym[order]]

        bars = ax.bar(range(len(feature_names)), values_sym[order],
                      color=colors, alpha=0.9, edgecolor='black', linewidth=0.8)

        ax.set_xticks(range(len(feature_names)))
        ax.set_xticklabels(np.array(feature_names)[order], rotation=45, ha='right', fontsize=11)
        ax.set_ylabel("Attribution", fontsize=13, fontweight='bold')
        ax.set_title(f"Feature Attribution Comparison — {title_base}\n(Bar: global Symmetric Shapley)",
                    fontsize=13, fontweight='bold')
        ax.axhline(y=0, color='black', linestyle='-', linewidth=1.2)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        plt.tight_layout()
        plt.savefig(fname_sym, dpi=150, bbox_inches='tight', facecolor='white')
        plt.close()
        print(f"Saved: {fname_sym}")

        # Plot 2: ASV
        fig, ax = plt.subplots(figsize=(10, 6))
        order = np.argsort(values_asv)[::-1]
        colors = ['#1f77b4' if v >= 0 else '#d62728' for v in values_asv[order]]

        bars = ax.bar(range(len(feature_names)), values_asv[order],
                      color=colors, alpha=0.9, edgecolor='black', linewidth=0.8)

        ax.set_xticks(range(len(feature_names)))
        ax.set_xticklabels(np.array(feature_names)[order], rotation=45, ha='right', fontsize=11)
        ax.set_ylabel("Attribution", fontsize=13, fontweight='bold')
        ax.set_title(f"Feature Attribution Comparison — {title_base}\n(Bar: global ASV)",
                    fontsize=13, fontweight='bold')
        ax.axhline(y=0, color='black', linestyle='-', linewidth=1.2)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        plt.tight_layout()
        plt.savefig(fname_asv, dpi=150, bbox_inches='tight', facecolor='white')
        plt.close()
        print(f"Saved: {fname_asv}")

    comparison_png = f"outputs/{out_prefix}_comparison_onmanifold.png"
    plot_comparison(global_sym, global_asv, global_sym_std, global_asv_std,
                   comparison_png, f"{out_prefix}: On-Manifold Comparison")

    sym_single_png = f"outputs/{out_prefix}_symmetric_single.png"
    asv_single_png = f"outputs/{out_prefix}_asv_single.png"
    plot_combined_comparison(global_sym, global_asv, sym_single_png, asv_single_png, out_prefix)

    # Console summary
    print(f"\n{'='*80}")
    print("RESULTS SUMMARY")
    print(f"{'='*80}")

    print(f"\nTop features (Symmetric on-manifold):")
    for i in np.argsort(global_sym)[::-1][:10]:
        print(f"  {feature_names[i]:25s} {global_sym[i]:+.6f} ± {global_sym_std[i]:.6f}")

    print(f"\nTop features (ASV on-manifold):")
    for i in np.argsort(global_asv)[::-1][:10]:
        print(f"  {feature_names[i]:25s} {global_asv[i]:+.6f} ± {global_asv_std[i]:.6f}")

    # Highlight differences
    print(f"\nLargest ASV vs Symmetric differences:")
    diff = global_asv - global_sym
    for i in np.argsort(np.abs(diff))[::-1][:5]:
        print(f"  {feature_names[i]:25s} Δ = {diff[i]:+.6f}")


def main():
    """Main execution function"""
    print("\n" + "="*80)
    print("Complete Paper-Faithful Asymmetric Shapley Values (ASV)")
    print("Based on arXiv:1910.06358v3 - Frye, Feige, & Rowat (2020)")
    print("="*80)

    # Define datasets to analyze
    tasks = [
        ("../../datasets/lung_cancer_risk.csv", "lung_cancer_risk",
         DATASET_CONSTRAINTS.get('lung_cancer_risk', {}),
         "lung_cancer_rf", "regression"),
        ("../../datasets/cv_risk.csv", "cv_risk",
         DATASET_CONSTRAINTS.get('cv_risk', {}),
         "cv_risk_rf", "regression"),
    ]

    for csv_path, target, constraints, prefix, task_type in tasks:
        if os.path.exists(csv_path):
            try:
                run_for_dataset(csv_path, target, constraints, prefix, task_type)
            except Exception as e:
                print(f"\nERROR processing {csv_path}: {e}")
                import traceback
                traceback.print_exc()
        else:
            print(f"\nWarning: {csv_path} not found, skipping...")

    print("\n" + "="*80)
    print("Analysis complete! Check the 'outputs/' directory for results.")
    print("="*80)

if __name__ == "__main__":
    main()


Complete Paper-Faithful Asymmetric Shapley Values (ASV)
Based on arXiv:1910.06358v3 - Frye, Feige, & Rowat (2020)

DATASET: ../datasets/lung_cancer_risk.csv
Target: lung_cancer_risk | Task: regression

Training model...
Train R²: 0.9643 | Test R²: 0.7291

Creating causal DAG...
Causal edges: [('smoking', 'stress'), ('smoking', 'drink_coffee')]

Initializing on-manifold imputer...

Evaluating on 50 test points...
  Processing point 1/50
  Processing point 11/50
  Processing point 21/50
  Processing point 31/50
  Processing point 41/50

Aggregating to global values...

Efficiency verification:
  Mean gap (Symmetric): 0.000000
  Mean gap (ASV):       0.000000
  (Should be close to 0 - verifies Axiom 1)

Saved: outputs/lung_cancer_rf_global_symmetric_onmanifold.csv
Saved: outputs/lung_cancer_rf_global_asv_onmanifold.csv
Saved: outputs/lung_cancer_rf_comparison_onmanifold.png
Saved: outputs/lung_cancer_rf_symmetric_single.png
Saved: outputs/lung_cancer_rf_asv_single.png

RESULTS SUMMARY

T